#### Librerías

In [ ]:
from zipfile import ZipFile

In [ ]:
import datetime as dt

In [ ]:
import win32com.client as win32

In [ ]:
import pandas as pd

In [ ]:
import openpyxl

In [ ]:
import os, shutil

In [ ]:
import win32con, win32api

In [ ]:
import logging

#### Vaciar la carpeta V:/BIVSA/ADMOPER/Cierre diario/Interfaz

In [ ]:
vaciar = 'V:/BIVSA/ADMOPER/Cierre diario/Interfaz' #Define la ruta.
for files in os.listdir(vaciar):
    path = os.path.join(vaciar, files)
    try:
        shutil.rmtree(path)
    except OSError:
        os.remove(path)

#### Vaciar la carpeta V:/BIVSA/ADMOPER/Cierre diario/Vectores

In [ ]:
vaciar_v = 'V:/BIVSA/ADMOPER/Cierre diario/Vectores' #Define la ruta.
for files in os.listdir(vaciar_v):
    path = os.path.join(vaciar_v, files)
    try:
        shutil.rmtree(path)
    except OSError:
        os.remove(path)

#### Definición de variables de fechas

In [ ]:
hoy = dt.date.today() #Determina fecha del día. 

In [ ]:
fa = (hoy.strftime('%Y%m%d')) #Determina la fecha del día para las rutas de los archivos.

#### Definiciones para el archivo de Log

In [ ]:
logger = logging.getLogger() #Llama a la función de logging.

In [ ]:
logger.setLevel(logging.DEBUG) #Define el nivel a partir del que se mostrarán los mensajes.

In [ ]:
log_local = 'C:/Temp/cierre_diario.log' #Archivo donde guarda el log localmente para evitar problemas de red.

In [ ]:
log_red = 'V:/BIVSA/ADMOPER/Cierre diario/Historial de interfaces/' + fa +'/cierre_diario.log' #Archivo final en la red.

In [ ]:
fhandler = logging.FileHandler(filename=log_local, mode='a') #Archivo donde guarda el log.

In [ ]:
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s') #Formato de las líneas de log.

In [ ]:
fhandler.setFormatter(formatter) #Setea los formatos.

In [ ]:
logger.addHandler(fhandler) #Agrega los formatos.

#### Usuarios permitidos

In [ ]:
usuarios = ('sm113925','cs214857','cr118369','ia214828')

#### Función Inicio del cierre

In [ ]:
def inicio_cierre():
    #Información para el archivo de log------------------------------------------------------------
    logging.info('----Generación de Precios Version 2.2 - Actualizado: 06/10/2025----') #Mensaje para el logging.
    logging.info('Usuario operador: ' + user) #Mensaje para el logging. 
    logging.info('Información descargada: ' + descarga) #Mensaje para el logging. 
    logging.info('Se inicia el proceso de consolidación de información.') #Mensaje para el logging.   
    print('Iniciando proceso de consolidación de información...') #Mensaje a mostrar en la consola.

#### Archivos para verificar actualizaciones

In [ ]:
def actualizaciones():
    archivos = [
        'V:/Alpha/PORTFOLI/Reuters/ACCIONES EN USD LATAM.xlsm',
        'V:/Alpha/PORTFOLI/Reuters/Precios Cedears 626 e Intereses Corridos 646.xlsm',
        'V:/BIVSA/ADMOPER/Cierre diario/VF/Reporte_Especies.xlsx'
    ]

    def obtener_ultima_actualizacion(ruta_archivo):
        try:
            timestamp = os.path.getmtime(ruta_archivo)
            fecha_hora = dt.datetime.fromtimestamp(timestamp)
            return fecha_hora.strftime("%Y-%m-%d %H:%M:%S")
        except FileNotFoundError:
            return "¡Archivo no encontrado!"
        except Exception as e:
            return f"Error: {str(e)}"

    for archivo in archivos:
        ultima_modificacion = obtener_ultima_actualizacion(archivo)
        print(f"Archivo {archivo} actualizado el {ultima_modificacion}")
        #print(f"Última modificación: {ultima_modificacion}\n")
        logging.info(f"Archivo {archivo} actualizado el {ultima_modificacion}") #Mensaje para el logging. 

#### Función extrae y envía vectores

In [ ]:
def vectores():
    #Extrae los archivos del zip de vectores CAFCI--------------------------------------------
    filev = 'V:/BIVSA/ADMOPER/Cierre diario/' + fa + "_Vectores_CLE.zip" #Crea la ruta.
    with ZipFile(filev, 'r') as zObject:
        zObject.extractall( 
            path='V:/BIVSA/ADMOPER/Cierre diario/Vectores')
    print('1-Se extrajeron los archivos de Vectores CAFCI!') #Mensaje a mostrar en la consola.
    logging.info('Se extrajeron exitosamente los Vectores CAFCI.') #Mensaje para el logging.
    #-----------------------------------------------------------------------------------------
    
    #Envio por e-mail con los Vectores------------------------------------------------------------------------
    outlook = win32.Dispatch('outlook.application')
    mail = outlook.CreateItem(0)
    mail.To= 'jose.aristi@icbc.com.ar; sergio.maugeri@icbc.com.ar; leandro.szymanski@icbc.com.ar; leonardo.lopez@icbc.com.ar; ruben.llambi@icbc.com.ar; daiana.andres@icbc.com.ar ;natalia.espinola@icbc.com.ar; sol.ponce@icbc.com.ar'
    mail.CC= 'alphafondosdeinversion@icbc.com.ar'
    mail.Subject = 'Vectores ' + fa
    mail.HTMLBody = 'Estimados:' + '<br/><br/>'
    mail.HTMLBody = mail.HTMLBody + 'Se adjunta un archivo con los vectores del día.'+ '<br/><br/>'
    mail.HTMLBody = mail.HTMLBody + 'Saludos!'
    mail.Attachments.Add(filev)
    mail.Send()
    
    print('2-Vectores enviados por correo electrónico!') #Mensaje a mostrar en la consola.
    logging.info('Se enviaron por e-mail los Vectores CAFCI a los Portfolio Managers.') #Mensaje para el logging. 

#### Función Precios Acciones U$S y Bloomberg

In [ ]:
def acc_usd_bbg():
    #---Armado del archivo Precio Acciones U$S y Bloomberg Interfaz--------------------------------------------------------
    #Lectura del archivo de Portfolios Precios Acciones U$S
    precios_USD = pd.read_excel("V:/Alpha/PORTFOLI/Reuters/ACCIONES EN USD LATAM.xlsm", sheet_name="Copia Valores", engine='openpyxl') #Importa los datos del archivo.
    columnas_a_eliminar = [' ar equity', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10'] #Determina las columnas a eliminar.
    precios_USD = precios_USD.drop(columns=columnas_a_eliminar, errors='ignore') #Elimina las columnas e ignora errores
    precios_USD = precios_USD.rename(columns={'Unnamed: 1':'RIC','Unnamed: 4':'PRECIO'}) #Renombra las columnas.
    precios_USD = precios_USD.iloc[1:-3] #Elimina primera fila (encabezado viejo) y últimas 3 filas
    pos_precio = precios_USD.columns.get_loc('PRECIO') #Busca la ubicación de la columna Precio, para tener de referencia.
    precios_USD.insert(pos_precio + 1, 'RUEDA', 'BBG') #Inserta la columna de la Rueda.
    precios_USD.insert(pos_precio + 2, 'MONEDA', 'U$S') #Inserta la columna Fecha con la fecha del día.    

    #Lectura del archivo de Portfolios Bloomberg
    bbg = pd.read_excel ("V:/Alpha/PORTFOLI/Reuters/Precios Cedears 626 e Intereses Corridos 646.xlsm", sheet_name="Actual",usecols=[' ', 'Last    ',' Chg    ','Px'], engine='openpyxl') #Importa los datos del archivo
    bbg = bbg.rename (columns={' ':'RIC','Last    ':'PRECIO',' Chg    ':'RUEDA','Unnamed: 3':'FECHA','Px':'MONEDA'}) #Renombra las columnas.
    bbg = bbg.dropna(subset=['MONEDA']) #Filta por el campo Moneda aquellas filas que no tienen ese valor.

    #Lectura del archivo de Portfolios para activos con precio en USD local
    cafci = pd.read_excel ('V:/Alpha/PORTFOLI/Reuters/Precios Cedears 626 e Intereses Corridos 646.xlsm', sheet_name='CAFCI') #Importa los datos del archivo
    precios_usd = pd.read_excel ('V:/BIVSA/ADMOPER/Cierre diario/Vectores/' + fa + ' CAFCI Vectores - Precios C.xlsx', skiprows=[0]) #Importa los datos del archivo.
    precios_usd = precios_usd.rename (columns={'Unnamed: 0':'Denominación', 'Valuación':'MONEDA'}) #Renombra las columnas.
    precios_usd.MONEDA = precios_usd.MONEDA.replace({'ARS':'$', 'USC':'U$S','USD':'U$S', 'USM':'U$S', 'BRL':'R$'}) #Reemplaza por los símbolos de moneda.
    cols_usd =[0,1,2,14,15,16,17] #Columnas necesarias pora los demás activos.
    precios_usd = precios_usd.iloc[:, cols_usd] #Selecciona las columnas para los demás activos.
    #Arma las tablas de precios
    cafci_precios_usd = pd.merge(cafci, precios_usd, on = 'BYMA') #Junta las dos tablas.
    #Arma la tabla de T+0
    cafci_precios_usd_t0 = cafci_precios_usd.drop(['BYMA','Denominación','ISIN','24 hs.','48 hs.'], axis=1) #Elimino columnas que no necesito para T+0.
    insertar = cafci_precios_usd_t0.columns.get_loc('Cdo.') #Busca la ubicación de la columnas 3, para tener de referencia.
    cafci_precios_usd_t0.insert(insertar + 1,'RUEDA', 'USDT+0') #Inserta la columna de la Rueda.
    cafci_precios_usd_t0 = cafci_precios_usd_t0.rename (columns={'Cdo.':'PRECIO'}) #Renombra la columna.
    #Arma la tabla de T+1
    cafci_precios_usd_t1 = cafci_precios_usd.drop(['BYMA','Denominación','ISIN','Cdo.','48 hs.'], axis=1) #Elimino columnas que no necesito para T+0.
    insertar1= cafci_precios_usd_t1.columns.get_loc('24 hs.') #Busca la ubicación de la columnas 3, para tener de referencia.
    cafci_precios_usd_t1.insert(insertar1 + 1,'RUEDA', 'USDT+1') #Inserta la columna de la Rueda.
    cafci_precios_usd_t1 = cafci_precios_usd_t1.rename (columns={'24 hs.':'PRECIO'}) #Renombra la columna.
    #Junta las dos tabla
    precios_cafci_usd =  pd.concat([cafci_precios_usd_t0, cafci_precios_usd_t1], axis=0) #Crea la tabla total con el contenido de ambas tablas.
    precios_cafci_usd = precios_cafci_usd.rename (columns={'BBG':'RIC'}) #Renombra las columnas.
    precios_cafci_usd = precios_cafci_usd [['RIC', 'PRECIO','RUEDA','MONEDA']] #Ordena las columnas.
    insertar2 = precios_cafci_usd.columns.get_loc('RUEDA') #Busca la ubicación de la columna Rueda, para tener de referencia.
    #precios_cafci_usd.insert(insertar2 + 1, 'FECHA', (hoy.strftime('%d/%m/%Y'))) #Inserta la columna Fecha con la fecha del día y su formato.
    #precios_cafci_usd = precios_cafci_usd.sort_values(by=['RIC','RUEDA'], ascending=True) #Ordena las columnas por el instrumento.
    
    #Junta los tres archivos
    total = pd.concat([precios_USD, bbg, precios_cafci_usd], axis=0) #Crea la tabla total con el contenido de ambas tablas.
    total = total.sort_values(by=['RIC','RUEDA'], ascending=True) #Ordena las columnas por el instrumento.
    pos_rueda = total.columns.get_loc('RUEDA') #Busca la ubicación de la columna Rueda, para tener de referencia.
    total.insert(pos_rueda + 1, 'FECHA', (hoy.strftime('%d/%m/%Y'))) #Inserta la columna Fecha con la fecha del día y su formato.

    #Devuelve el DataFrame para usarlo en otra función
    return total

#### Función Exportación Precios Acciones U$S y Bloomberg

In [ ]:
def export_acc_usd_bbg():    
    total = acc_usd_bbg() # Convierte en un Dataframe.
    ruta = 'V:/BIVSA/ADMOPER/Cierre diario/Interfaz/Precios Acciones U$S y Bloomberg Interfaz.xlsx' #Define el nombre y ruta del archivo.
    total.to_excel(ruta, sheet_name='Hoja1', index=False) #Convierte y exporta la tabla total a Excel.
    print('3-Precio Acciones U$S y Bloomberg Interfaz generado en', ruta) #Mensaje a mostrar en la consola.
    logging.info('Se genera ' + ruta) #Mensaje para el logging.

#### Función Tipo de Cambio

In [ ]:
def tc():
    #---Armado del archivo con los tipos de cambio---------------------------------------------------------
    #Lectura del archivo de Portfolios Bloomberg
    blg = pd.read_excel ('V:/BIVSA/ADMOPER/Cierre diario/Interfaz/Precios Acciones U$S y Bloomberg Interfaz.xlsx',usecols=['RIC','PRECIO'], engine='openpyxl') #Importa los datos del archivo.
    blg = blg.loc[blg['RIC'] == 'REAL'] #Filtra para quedarse con la cotización del Real.

    #Lectura del archivo de Variables de CAFCI
    file = 'V:/BIVSA/ADMOPER/Cierre diario/Vectores/' + fa + ' CAFCI Vectores - Variables C.xlsx' #Crea la ruta.
    TCC = pd.read_excel (file, usecols=['Código Bloomberg', 'Last'], engine='openpyxl') #Importa los datos del archivo.
    TCC= TCC.set_axis(['Moneda', 'Valor'], axis=1) #Renombro las columnas.
    TCC = TCC.loc[TCC['Moneda'] == 'ARS MAEF curncy'] #Filtra para quedarme sólo con la cotización del USD.
    Fecha=(hoy.strftime('%d/%m/%Y')) #Determina la fecha del día.

    TC_Real = float(TCC['Valor'].iloc[0] / blg['PRECIO'].iloc[0]) #Genera el valor del Tipo de cambio del Real y lo pasa a números (float).
    TC_USD = float(TCC['Valor'].iloc[0]) #Genera el valor del Tipo de cambio del USD y lo pasa a números (float).

    #Genera el dataset con la información necesario
    data = {
        'Moneda desde': ['R$', 'U$S', 'U$C', 'U$M', 'R$M'],
        'Moneda hasta': ['$', '$', '$', '$', '$'],
        'TC comprador': [TC_Real, TC_USD, TC_USD, TC_USD, TC_Real],
        'TC Vendedor': [TC_Real, TC_USD, TC_USD, TC_USD, TC_Real],
        'Fecha': [Fecha,Fecha,Fecha,Fecha,Fecha]
    }

    data = pd.DataFrame(data) #Convierte la tabla en un DataFrame.
    
    #Exporta el archivo
    ruta1 = 'V:/BIVSA/ADMOPER/Cierre diario/Interfaz/TC.xlsx' #Define el nombre y la ruta del archivo.
    data.to_excel(ruta1, sheet_name='Hoja1', index=False) #Convierte y exporta la tabla a Excel.
    print('4-TC generado en', ruta1) #Mensaje a mostrar en la consola.
    logging.info('Se genera ' + ruta1) #Mensaje para el logging. 

#### Función Precios CAFCI

In [ ]:
def precios_cafci():
    #---Armado del archivo Precios CAFCI Interfaz-------------------------
    #Importa las especies de VF
    especies = pd.read_excel ('V:/BIVSA/ADMOPER/Cierre diario/VF/Reporte_Especies.xlsx', skiprows=[0,2,3], engine='openpyxl') #Importa los datos del archivo.
    especies = especies.drop(['Moneda','Código'], axis=1) #Elimino columnas que no voy a usar.
    especies = especies.rename (columns={'ISIN Code':'ISIN', 'Abreviatura por Defecto':'RIC'}) #Renombra las columnas.
    especies = especies.dropna(subset=['ISIN']) #Filta por el campo aquellas filas que no tienen ese valor.    

    #Importa el archivo de CAFCI Precios C
    file1 = 'V:/BIVSA/ADMOPER/Cierre diario/Vectores/' + fa + ' CAFCI Vectores - Precios C.xlsx' #Crea la ruta.
    precios = pd.read_excel (file1, skiprows=[0], engine='openpyxl') #Importa los datos del archivo.
    precios = precios.rename (columns={'Unnamed: 0':'Denominación', 'Valuación':'MONEDA'}) #Renombra las columnas.
    precios.MONEDA = precios.MONEDA.replace({'ARS':'$', 'USC':'U$S','USD':'U$S', 'USM':'U$S', 'BRL':'R$'}) #Reemplaza por los símbolos de moneda.

    #Selecciona las columnas para cada tabla
    cols_fut = [0,4,14,15] #Columnas necesarias pora los futuros.
    cols_cafci =[0,1,2,14,15,16,17] #Columnas necesarias pora los demás activos.

    #Arma las tablas de futuros y de Precios
    futuros=precios.iloc[:, cols_fut] #Selecciona las columnas para futuros.
    precios = precios.iloc[:, cols_cafci] #Selecciona las columnas para los demás activos.

    #Tabla de futuros
    futuros = futuros[futuros['CAFCI'].str.startswith('FUTDLR', na=False)] #Filta para quedarse con las cotizaciones de los futuros según el códo CAFCI.
    futuros = futuros.rename (columns={'CAFCI':'ISIN'}) #Renombra las columnas.
    futuros.insert(2, 'BYMA', futuros['ISIN'].str.slice(3)) #Agrega la columna BYMA copiando los datos de ISIN, sin los primeros tres caractéres.
    futuros.insert(5, '24 hs.', futuros['Cdo.']) #Agrega la columna 24 hs. copiando los datos de Cdo.
    futuros.insert(6, '48 hs.', futuros['Cdo.']) #Agrega la columna BYMA copiando los datos de ISIN.

    #Unifica la tabla de Futuros y de Precios
    precios = pd.concat([precios, futuros], axis=0) # Consolida las tablas Precios y Futuros en una sola.
    
    #Arma las tablas de precios
    especies_precios = pd.merge(especies, precios, on = 'ISIN') #Junta las dos tablas.
    especies_precios = especies_precios.dropna(subset=['RIC']) #Filta por el campo aquellas filas que no tienen ese valor.

    #Arma la tabla de T+0
    ep0 = especies_precios.drop(['ISIN','Descripción', 'Denominación','BYMA','24 hs.','48 hs.'], axis=1) #Elimino columnas que no necesito para T+0.
    ep0 = ep0.dropna(subset=['Cdo.']) #Filta por el campo Cdo. aquellas filas que no tienen ese valor.
    ep0 = ep0.drop_duplicates(subset=['RIC'], keep='first') #Elimina filas duplicadas, por tener varias coincidencias en el ISIN.
    pos_columna_a_insertar2 = ep0.columns.get_loc('Cdo.') #Busca la ubicación de la columnas 3, para tener de referencia.
    ep0.insert(pos_columna_a_insertar2 + 1,'RUEDA', 'T+0') #Inserta la columna de la Rueda.
    ep0 = ep0.rename (columns={'Cdo.':'PRECIO'}) #Renombra la columna.

    #Arma la tabla de T+1
    ep1 = especies_precios.drop(['ISIN','Descripción', 'Denominación','BYMA','Cdo.','48 hs.'], axis=1) #Elimino columnas que no necesito para T+1.
    ep1 = ep1.dropna(subset=['24 hs.']) #Filta por el campo Cdo. aquellas filas que no tienen ese valor.
    ep1 = ep1.drop_duplicates(subset=['RIC'], keep='first') #Elimina filas duplicadas, por tener varias coincidencias en el ISIN.
    pos_columna_a_insertar3 = ep1.columns.get_loc('24 hs.') #Busca la ubicación de la columnas 3, para tener de referencia.
    ep1.insert(pos_columna_a_insertar3 + 1,'RUEDA', 'T+1') #Inserta la columna de la Rueda.
    ep1 = ep1.rename (columns={'24 hs.':'PRECIO'}) #Renombra la columna.

    #Junta las dos tablas
    precios_cafci =  pd.concat([ep0, ep1], axis=0) #Crea la tabla total con el contenido de ambas tablas.
    pos_columna_a_insertar4 = ep1.columns.get_loc('RUEDA') #Busca la ubicación de la columna 3 para tener de referencia.
    precios_cafci.insert(pos_columna_a_insertar4 + 1,'FECHA', (hoy.strftime('%d/%m/%Y'))) #Inserta la columna Fecha con la fecha del día y su formato.
    precios_cafci = precios_cafci [['RIC', 'PRECIO','RUEDA','FECHA','MONEDA']] #Ordena las columnas.
    precios_cafci = precios_cafci.sort_values(by=['RIC','RUEDA'], ascending=True) #Ordena las columnas por el instrumento.

    #Devuelve el DataFrame para usarlo en otra función 
    return {"especies": especies, "precios_cafci": precios_cafci}

#### Función Exportación Precios CAFCI

In [ ]:
def export_precios_cafci():
    tablas = precios_cafci() # Convierte en un Dataframe la salida de la función.
    cafci = tablas["precios_cafci"] # Convierte en un Dataframe la tabla.
    ruta2 = 'V:/BIVSA/ADMOPER/Cierre diario/Interfaz/Precios CAFCI Interfaz.xlsx' #Define el nombre y ruta del archivo.
    cafci.to_excel(ruta2, sheet_name='Hoja1', index=False) #Convierte y exporta la tabla total a Excel.
    print('5-Precios CAFCI Interfaz generado en', ruta2) #Mensaje a mostrar en la consola.
    logging.info('Se genera ' + ruta2) #Mensaje para el logging. 

#### Función Tabla especies sin ISIN

In [ ]:
def sin_isin():
    #---Arma la tabla para la lista de especies sin ISIN----------------------------------------------------
    #Arma la tabla principal
    varios = pd.read_excel ('V:/BIVSA/ADMOPER/Cierre diario/VF/Reporte_Especies.xlsx', skiprows=[0,2,3], engine='openpyxl') #Importa los datos del archivo.
    varios = varios.drop(['Moneda','Código'], axis=1) #Elimino columnas que no voy a usar
    varios = varios.rename (columns={'ISIN Code':'ISIN', 'Abreviatura por Defecto':'RIC'}) #Renombra las columnas.
    varios = varios[varios['ISIN'].isnull()] # Filtra para quedarse con los campos que no tienen ISIN.    
    varios = varios.rename (columns={'Abreviatura por Defecto':'RIC'}) #Renombra la columna.
    varios = varios.drop(['ISIN','Descripción'], axis=1) #Elimina columnas que no necesito para T+0.
    pos_columna_a_insertar5 = varios.columns.get_loc('RIC') #Busca la ubicación de la columnas 1, para tener de referencia.
    varios.insert(pos_columna_a_insertar5 + 1,'Precio','') #Inserta la columna Fecha con la fecha del día y su formato.
    varios.insert(pos_columna_a_insertar5 + 2,'Rueda','T+0') #Inserta la columna Fecha con la fecha del día y su formato.
    varios.insert(pos_columna_a_insertar5 + 3,'FECHA', (hoy.strftime('%d/%m/%Y'))) #Inserta la columna Fecha con la fecha del día y su formato.

    #Arma la lista en T+1
    varios1 = varios.copy()
    varios1 = varios1.replace({'T+0': 'T+1'})

    #Une las dos listas
    varios_total =  pd.concat([varios, varios1], axis=0) #Crea la tabla total con el contenido de las tres tablas.
    varios_total = varios_total.sort_values(by=['RIC','Rueda'], ascending=True) #Ordena las columnas por el instrumento.

    #Exporta el archivo de especies sin isin
    ruta3 = 'V:/BIVSA/ADMOPER/Cierre diario/Interfaz/Lista sin isin.xlsx' #Define el nombre y ruta del archivo.
    varios_total.to_excel(ruta3, index=False) #Convierte y exporta la tabla total a Excel.
    print('6-Lista de especies sin Isin generada en', ruta3) #Mensaje a mostrar en la consola.
    logging.info('Se genera ' + ruta3) #Mensaje para el logging. 

#### Función Tabla precios consolidados

In [ ]:
def precios_consolidados():
    # Ejecuta las funciones que generan y devuelven las tablas individuales
    df_acc_usd_bbg = acc_usd_bbg()  #DataFerame de Acciones USD y Bloomberg.
    datos = precios_cafci()  # DataFerame Precios Cafci y de Especies.
    df_precios_cafci = datos["precios_cafci"] #Separa la tabla de Precios Cafci.
    df_especies = datos ["especies"] #Separa la tabla de Especies.
    
    precios_todos = pd.concat([df_acc_usd_bbg, df_precios_cafci], axis=0) #Concatena las tablas.
           
    control_ric = pd.merge(df_especies, precios_todos, on = 'RIC', how = 'outer') #Junta las dos tabla.
    #control_ric = control_ric[control_ric['PRECIO'].isnull()]  

    ruta4 = 'V:/BIVSA/ADMOPER/Cierre diario/Interfaz/Lista de precios.xlsx' #Define el nombre y ruta del archivo.
    control_ric.to_excel(ruta4, index=False) #Convierte y exporta la tabla total a Excel.
    print('7-Lista de precios generada en', ruta4) #Mensaje a mostrar en la consola.
    logging.info('Se consolidan todos los precios con todas las especies.') #Mensaje para el logging. 

#### Función copia y bloqueo de archivos

In [ ]:
def bloqueo_archivos():
    #---Back up de la información generada-------------------------------    
    origen='V:/BIVSA/ADMOPER/Cierre diario/Interfaz/'
    destino='V:/BIVSA/ADMOPER/Cierre diario/Historial de interfaces/' + fa
    shutil.copytree(origen, destino)
    
    #---Bloque de los archivos-------
    ro0='V:/BIVSA/ADMOPER/Cierre diario/Historial de interfaces/' + fa +'/Precios Acciones U$S y Bloomberg Interfaz.xlsx' #Define el nombre y ruta del archivo.
    win32api.SetFileAttributes(ro0, win32con.FILE_ATTRIBUTE_READONLY) #Configura el archivo como sólo lectura.
    ro1='V:/BIVSA/ADMOPER/Cierre diario/Historial de interfaces/' + fa +'/TC.xlsx' #Define el nombre y ruta del archivo.
    win32api.SetFileAttributes(ro1, win32con.FILE_ATTRIBUTE_READONLY) #Configura el archivo como sólo lectura.
    ro2='V:/BIVSA/ADMOPER/Cierre diario/Historial de interfaces/' + fa +'/Precios CAFCI Interfaz.xlsx' #Define el nombre y ruta del archivo.
    win32api.SetFileAttributes(ro2, win32con.FILE_ATTRIBUTE_READONLY) #Configura el archivo como sólo lectura.
    ro3='V:/BIVSA/ADMOPER/Cierre diario/Historial de interfaces/' + fa +'/Lista sin isin.xlsx' #Define el nombre y ruta del archivo.
    win32api.SetFileAttributes(ro3, win32con.FILE_ATTRIBUTE_READONLY) #Configura el archivo como sólo lectura.
    
    logging.info('Se realiza el backup de la información generada.') #Mensaje para el logging. 
    logging.info('Fin del proceso.') #Mensaje para el logging.

#### Función Log de cierre

In [ ]:
def cierre_log():
    shutil.copy(log_local, log_red) #Copia el log del arvhivo local a la red.
    fhandler.close() #Libera el archivo de log.
    logger.removeHandler(fhandler) #Elimina el handler
    if os.path.exists(log_local): #Elimina el archivo local.
        os.remove(log_local)
    print('Información de cierre generada satisfactoriamente!') #Mensaje a mostrar en la consola.

#### Función Usuario válido

In [ ]:
def usuario_valido():
    inicio_cierre()
    actualizaciones()
    vectores()
    acc_usd_bbg()
    export_acc_usd_bbg()
    tc()
    precios_cafci()
    export_precios_cafci()
    sin_isin()
    precios_consolidados()
    bloqueo_archivos()
    cierre_log()

#### Función Usuario inválido

In [ ]:
def usuario_invalido():
    print("🛑Acceso denegado🛑 Usuario inválido.")
    outlook = win32.Dispatch('outlook.application')
    mail = outlook.CreateItem(0)
    mail.To = 'alphafondosdeinversion@icbc.com.ar' 
    mail.Subject = '🚨ATENCION: Acceso no permitido a Generación de Precios cierre diario!!'
    mail.HTMLBody = 'Un usuario no autorizado ['+ user +'] intentó acceder al archivo de cierre diario.' + '<br/><br/>'
    mail.Send()

#### Ejecución

In [ ]:
print('----Generación de Precios Version 2.2 - Actualizado: 06/10/2025----')
user=os.getenv('USERNAME')
user=user.lower()
print("🔑Usuario detectado:", user)

if user in usuarios:
    descarga = input("🧩¿Descargó la información necesaria: -Vectores CAFCI, -VF: Reporte de especies? (y/n): ")
    descarga = descarga.lower()
    if descarga =="y":
        usuario_valido()
    else:
        print('⛔Descargue la información y vuelva a intentarlo.')
else:
    usuario_invalido()